In [5]:
"""
PRISM-AI pipeline repeated over many random seeds (default 25).

For every seed the script re-draws the grouped 80/20 train/test split and re-seeds every model, then
reports mean ± SD across seeds for Tables V, VI, VII, VIII, IX and mean ROC curves (± 1 SD band).
The 2,000-project dataset itself (prism_synthetic_dataset.csv) stays fixed.

Run:   python prism_multiseed.py            # 25 seeds, uses all CPU cores
       python prism_multiseed.py 30 4       # 30 seeds, 4 worker processes
Needs: numpy pandas scikit-learn matplotlib
"""
import os
import sys
import time
import warnings
from concurrent.futures import ProcessPoolExecutor, as_completed

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.base import clone
from sklearn.compose import TransformedTargetRegressor
from sklearn.ensemble import (GradientBoostingClassifier, GradientBoostingRegressor,
                              RandomForestClassifier, RandomForestRegressor)
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import (accuracy_score, brier_score_loss, f1_score, mean_absolute_error,
                             mean_squared_error, precision_recall_curve, precision_score,
                             recall_score, roc_auc_score, roc_curve)
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC, SVR
from sklearn.tree import DecisionTreeClassifier

warnings.filterwarnings("ignore")
CSV = "prism_synthetic_dataset.csv"
OUT = "prism_outputs_multiseed"
N_SEEDS = 25
N_WORKERS = 1
SEEDS = list(range(1, N_SEEDS + 1))
STAGES = [0.2, 0.3, 0.4, 0.5, 0.6]
FPRS = [0.025, 0.05, 0.10, 0.20]
GRID = np.linspace(0, 1, 201)                       # common FPR grid for averaging ROC curves
NUM = ["progress_gap", "contractor_rating", "land_acquired", "milestone_delay_ratio",
       "original_cost_log", "financial_progress", "cost_progress_divergence",
       "contractor_past_delays", "weather_index", "elapsed_share"]
EW_MODELS = ["EVM index rule", "Logistic regression", "Random forest", "Proposed GBDT"]
T6_MODELS = ["EVM index rule", "Logistic regression", "SVM (RBF)", "MLP", "Random forest", "Proposed GBDT"]


# ----------------------------------------------------------------------------
# models (every random_state is the seed of the current repetition)
# ----------------------------------------------------------------------------
def classifiers(seed):
    return {
        "Logistic regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
        "Naive Bayes": GaussianNB(),
        "k-NN": make_pipeline(StandardScaler(), KNeighborsClassifier(25)),
        "Decision tree": DecisionTreeClassifier(max_depth=6, min_samples_leaf=20, random_state=seed),
        "SVM (RBF)": make_pipeline(StandardScaler(), SVC(probability=True, random_state=seed)),
        "MLP": make_pipeline(StandardScaler(), MLPClassifier((64, 32), alpha=1e-3, early_stopping=True,
                                                             max_iter=500, random_state=seed)),
        "Random forest": RandomForestClassifier(300, min_samples_leaf=5, n_jobs=1, random_state=seed),
        "Proposed GBDT": GradientBoostingClassifier(n_estimators=250, learning_rate=0.05, max_depth=3,
                                                    subsample=0.8, random_state=seed),
    }


def regressors(seed):
    def wrap(m):
        return TransformedTargetRegressor(make_pipeline(StandardScaler(), m), transformer=StandardScaler())
    return {
        "Linear (ridge)": wrap(Ridge(alpha=1.0)),
        "SVR (RBF)": wrap(SVR(C=3.0)),
        "Random forest": RandomForestRegressor(300, min_samples_leaf=5, n_jobs=1, random_state=seed),
        "Proposed GBDT": GradientBoostingRegressor(n_estimators=300, learning_rate=0.05, max_depth=3,
                                                   subsample=0.8, random_state=seed),
    }


def best_f1_threshold(y, p):
    prec, rec, thr = precision_recall_curve(y, p)
    f1 = 2 * prec[:-1] * rec[:-1] / np.clip(prec[:-1] + rec[:-1], 1e-9, None)
    return thr[np.argmax(f1)]


def metrics(y, p, thr):
    yhat = (p >= thr).astype(int)
    return dict(Acc=accuracy_score(y, yhat), Prec=precision_score(y, yhat, zero_division=0),
                Rec=recall_score(y, yhat), F1=f1_score(y, yhat), AUC=roc_auc_score(y, p))


def rmse(y, p):
    return np.sqrt(mean_squared_error(y, p))


# ----------------------------------------------------------------------------
# one full repetition of the pipeline
# ----------------------------------------------------------------------------
def run_seed(seed):
    t0 = time.time()
    data = pd.read_csv(CSV)
    X = pd.concat([data[NUM], pd.get_dummies(data["sector"], prefix="sector", dtype=float)], axis=1)
    tr_idx, te_idx = next(GroupShuffleSplit(1, test_size=0.2, random_state=seed).split(X, groups=data["pid"]))
    Xtr, Xte = X.iloc[tr_idx].reset_index(drop=True), X.iloc[te_idx].reset_index(drop=True)
    dtr, dte = data.iloc[tr_idx].reset_index(drop=True), data.iloc[te_idx].reset_index(drop=True)
    gtr = dtr["pid"].values
    gkf = GroupKFold(n_splits=5)

    def classify(target, evm_tr, evm_te):
        ytr, yte = dtr[target].values, dte[target].values
        res, probs, thrs = {}, {}, {}
        aucs = [roc_auc_score(ytr[v], evm_tr[v]) for _, v in gkf.split(Xtr, ytr, gtr)]
        thr = best_f1_threshold(ytr, evm_tr)
        m = metrics(yte, evm_te, thr)
        m.update(Brier=np.nan, CV_AUC=np.mean(aucs))
        res["EVM index rule"], probs["EVM index rule"], thrs["EVM index rule"] = m, evm_te, thr
        for name, model in classifiers(seed).items():
            oof, aucs = np.zeros(len(ytr)), []
            for a, v in gkf.split(Xtr, ytr, gtr):
                p = clone(model).fit(Xtr.iloc[a], ytr[a]).predict_proba(Xtr.iloc[v])[:, 1]
                oof[v] = p
                aucs.append(roc_auc_score(ytr[v], p))
            thr = best_f1_threshold(ytr, oof)
            p_te = clone(model).fit(Xtr, ytr).predict_proba(Xte)[:, 1]
            m = metrics(yte, p_te, thr)
            m.update(Brier=brier_score_loss(yte, p_te), CV_AUC=np.mean(aucs))
            res[name], probs[name], thrs[name] = m, p_te, thr
        return pd.DataFrame(res).T, probs, thrs

    t5_delay, p_delay, thr_delay = classify("y_delay", 1 - dtr["SPI"].values, 1 - dte["SPI"].values)
    t5_cost, p_cost, _ = classify("y_cost", 1 - dtr["CPI"].values, 1 - dte["CPI"].values)

    # Table VII - regression
    reg = {}
    evm_d = np.clip(dte["duration"] * (1 / dte["SPI"] - 1), 0, None)
    evm_c = dte["orig_cost"] / dte["CPI"]
    reg["EVM extrapolation"] = [mean_absolute_error(dte.delay_months, evm_d), rmse(dte.delay_months, evm_d),
                                mean_absolute_error(dte.final_cost, evm_c), rmse(dte.final_cost, evm_c)]
    Xtr_r = Xtr.assign(duration=dtr["duration"], orig_cost=dtr["orig_cost"])
    Xte_r = Xte.assign(duration=dte["duration"], orig_cost=dte["orig_cost"])
    for name, model in regressors(seed).items():
        pd_ = clone(model).fit(Xtr_r, dtr["delay_months"]).predict(Xte_r)
        pc_ = clone(model).fit(Xtr_r, dtr["final_cost"]).predict(Xte_r)
        reg[name] = [mean_absolute_error(dte.delay_months, pd_), rmse(dte.delay_months, pd_),
                     mean_absolute_error(dte.final_cost, pc_), rmse(dte.final_cost, pc_)]
    t7 = pd.DataFrame(reg, index=["Delay MAE", "Delay RMSE", "Cost MAE", "Cost RMSE"]).T

    # Table IX - early warning
    ew = {}
    for name in EW_MODELS:
        flag = pd.DataFrame(dict(pid=dte.pid, stage=dte.elapsed_share, y=dte.y_delay, dur=dte.duration,
                                 f=(p_delay[name] >= thr_delay[name])))
        per = flag.groupby("pid").agg(y=("y", "first"), dur=("dur", "first"), any_f=("f", "any"))
        first = flag[flag.f].groupby("pid").stage.min()
        pos, neg = per[per.y == 1], per[per.y == 0]
        lead = ((1 - first.reindex(pos.index[pos.any_f])) * pos.dur[pos.any_f]).mean()
        ew[name] = dict(Detected=100 * pos.any_f.mean(),
                        At_20pct=100 * flag[(flag.stage == 0.2) & (flag.y == 1)].f.mean(),
                        Lead_months=lead, False_alarm=100 * neg.any_f.mean())
    t9 = pd.DataFrame(ew).T

    # Table VI - TPR at fixed FPR, Table VIII - AUC by stage
    t6 = pd.DataFrame({n: [roc_curve(dte.y_delay, p_delay[n])[1][roc_curve(dte.y_delay, p_delay[n])[0] <= f].max()
                           for f in FPRS] for n in T6_MODELS}, index=["FPR 2.5%", "5%", "10%", "20%"]).T
    t8 = pd.DataFrame({n: [roc_auc_score(dte.y_delay[(dte.elapsed_share == s).values],
                                         p_delay[n][(dte.elapsed_share == s).values]) for s in STAGES]
                       for n in T6_MODELS}, index=[f"{int(100 * s)}%" for s in STAGES]).T

    # ROC curves on a common FPR grid (for averaging across seeds)
    def curves(target, probs):
        out = {}
        for n in T6_MODELS:
            fpr, tpr, _ = roc_curve(dte[target], probs[n])
            out[n] = np.interp(GRID, fpr, tpr)
        return out

    return dict(seed=seed, secs=time.time() - t0, t5_delay=t5_delay, t5_cost=t5_cost, t6=t6, t7=t7,
                t8=t8, t9=t9, roc_delay=curves("y_delay", p_delay), roc_cost=curves("y_cost", p_cost))


# ----------------------------------------------------------------------------
# aggregation helpers
# ----------------------------------------------------------------------------
def stack(frames):
    return np.stack([f.values.astype(float) for f in frames])


def summarise(frames, nd=3):
    """Return (mean DataFrame, SD DataFrame, 'mean±SD' string DataFrame) across seeds."""
    arr = stack(frames)
    mean = pd.DataFrame(np.nanmean(arr, 0), index=frames[0].index, columns=frames[0].columns)
    sd = pd.DataFrame(np.nanstd(arr, 0, ddof=1), index=frames[0].index, columns=frames[0].columns)
    txt = mean.copy().astype(object)
    for i in mean.index:
        for c in mean.columns:
            txt.loc[i, c] = "–" if np.isnan(mean.loc[i, c]) else f"{mean.loc[i, c]:.{nd}f}±{sd.loc[i, c]:.{nd}f}"
    return mean, sd, txt


def save_table(name, frames, nd=3, title=""):
    mean, sd, txt = summarise(frames, nd)
    print(f"\n{title}  (mean ± SD over {len(frames)} seeds)")
    print(txt.to_string())
    txt.to_csv(f"{OUT}/{name}_mean_sd.csv")
    mean.round(nd + 1).to_csv(f"{OUT}/{name}_mean.csv")
    sd.round(nd + 1).to_csv(f"{OUT}/{name}_sd.csv")
    return mean, sd


if __name__ == "__main__":
    os.makedirs(OUT, exist_ok=True)
    print(f"Running {len(SEEDS)} seeds {SEEDS[0]}..{SEEDS[-1]} on {N_WORKERS} worker(s)", flush=True)
    results = []
    if N_WORKERS == 1:                                  # sequential: safest inside Jupyter / on Windows
        for k, s in enumerate(SEEDS, 1):
            r = run_seed(s)
            results.append(r)
            print(f"  seed {r['seed']:>3} done ({r['secs']:.0f}s)   [{k}/{len(SEEDS)}]", flush=True)
    else:
        with ProcessPoolExecutor(max_workers=N_WORKERS) as ex:
            futs = [ex.submit(run_seed, s) for s in SEEDS]
            for k, f in enumerate(as_completed(futs), 1):
                r = f.result()
                results.append(r)
                print(f"  seed {r['seed']:>3} done ({r['secs']:.0f}s)   [{k}/{len(SEEDS)}]", flush=True)
    results.sort(key=lambda r: r["seed"])

    # Table V - classification (delay and cost side by side)
    for key, label in [("t5_delay", "delay"), ("t5_cost", "cost")]:
        save_table(f"tableV_{label}", [r[key] for r in results], 3,
                   f"TABLE V - Classification, {label} risk (test set; threshold from out-of-fold F1)")
    save_table("tableVI_tpr_at_fpr", [r["t6"] for r in results], 3,
               "TABLE VI - TPR at fixed FPR (delay, test set)")
    save_table("tableVII_regression", [r["t7"] for r in results], 2,
               "TABLE VII - Regression (delay in months, cost in crore)")
    save_table("tableVIII_auc_by_stage", [r["t8"] for r in results], 3,
               "TABLE VIII - Delay ROC-AUC by monitoring stage")
    save_table("tableIX_early_warning", [r["t9"] for r in results], 1,
               "TABLE IX - Early warning (Detected / At 20% / False alarm in %, Lead in months)")

    # per-seed AUC log + how often the proposed GBDT beats the others
    for key, label in [("t5_delay", "delay"), ("t5_cost", "cost")]:
        auc = pd.DataFrame({r["seed"]: r[key]["AUC"] for r in results}).T
        auc.index.name = "seed"
        auc.round(4).to_csv(f"{OUT}/auc_per_seed_{label}.csv")
        wins = {m: int((auc["Proposed GBDT"] > auc[m]).sum()) for m in auc.columns if m != "Proposed GBDT"}
        print(f"\nSeeds (of {len(auc)}) where Proposed GBDT has higher {label} AUC than: " + ", ".join(
            f"{m} {w}" for m, w in wins.items()))

    # mean ROC curves with ±1 SD bands
    STYLE = {"EVM index rule": "#999999", "Logistic regression": "#4c78b8", "SVM (RBF)": "#5fb8b0",
             "MLP": "#b77fb8", "Random forest": "#4caf50", "Proposed GBDT": "#e8474c"}
    fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.8))
    for ax, (title, rk, tk) in zip(axes, [("Delay risk", "roc_delay", "t5_delay"),
                                           ("Cost-overrun risk", "roc_cost", "t5_cost")]):
        for n in T6_MODELS:
            curves = np.stack([r[rk][n] for r in results])
            m, s = curves.mean(0), curves.std(0, ddof=1)
            aucs = np.array([r[tk].loc[n, "AUC"] for r in results])
            ax.plot(GRID, m, color=STYLE[n], lw=2.2 if n == "Proposed GBDT" else 1.2,
                    label=f"{n} ({aucs.mean():.3f}±{aucs.std(ddof=1):.3f})")
            ax.fill_between(GRID, np.clip(m - s, 0, 1), np.clip(m + s, 0, 1), color=STYLE[n], alpha=0.12, lw=0)
        ax.plot([0, 1], [0, 1], "k:", lw=1)
        ax.set_title(f"{title} - mean ROC over {len(results)} seeds")
        ax.set_xlabel("False positive rate"); ax.set_ylabel("True positive rate")
        ax.legend(loc="lower right", fontsize=7.5, frameon=False); ax.grid(alpha=.2)
    fig.tight_layout(); fig.savefig(f"{OUT}/fig_roc_mean_over_seeds.png", dpi=200); plt.close(fig)

    # AUC by stage with SD error bars
    fig, ax = plt.subplots(figsize=(5.8, 4.3))
    arr = stack([r["t8"] for r in results])
    for i, n in enumerate(T6_MODELS):
        ax.errorbar([20, 30, 40, 50, 60], arr[:, i, :].mean(0), yerr=arr[:, i, :].std(0, ddof=1), marker="o",
                    ms=4, capsize=2, color=STYLE[n], lw=2 if n == "Proposed GBDT" else 1.2, label=n)
    ax.set_xlabel("Elapsed share of approved duration (%)"); ax.set_ylabel("ROC-AUC (delay), mean ± SD")
    ax.set_xticks([20, 30, 40, 50, 60]); ax.legend(fontsize=8, frameon=False); ax.grid(alpha=.2)
    fig.tight_layout(); fig.savefig(f"{OUT}/fig_auc_by_stage_mean_over_seeds.png", dpi=200); plt.close(fig)

    print(f"\nDone. Tables (CSV) and figures (PNG) saved in ./{OUT}/")

Running 25 seeds 1..25 on 1 worker(s)
  seed   1 done (391s)   [1/25]
  seed   2 done (388s)   [2/25]
  seed   3 done (375s)   [3/25]
  seed   4 done (452s)   [4/25]
  seed   5 done (444s)   [5/25]
  seed   6 done (453s)   [6/25]
  seed   7 done (463s)   [7/25]
  seed   8 done (456s)   [8/25]
  seed   9 done (433s)   [9/25]
  seed  10 done (489s)   [10/25]
  seed  11 done (368s)   [11/25]
  seed  12 done (232s)   [12/25]
  seed  13 done (264s)   [13/25]
  seed  14 done (219s)   [14/25]
  seed  16 done (223s)   [16/25]
  seed  17 done (224s)   [17/25]
  seed  18 done (224s)   [18/25]
  seed  19 done (216s)   [19/25]
  seed  20 done (232s)   [20/25]
  seed  21 done (263s)   [21/25]
  seed  22 done (253s)   [22/25]
  seed  23 done (277s)   [23/25]
  seed  24 done (268s)   [24/25]
  seed  25 done (265s)   [25/25]

TABLE V - Classification, delay risk (test set; threshold from out-of-fold F1)  (mean ± SD over 25 seeds)
                             Acc         Prec          Rec           F1 